In [ ]:
# CELL 1 - Imports
#
# Only the Python standard library is imported into the notebook kernel.
# Every heavy step (dlib face landmarks, AV-HuBERT feature extraction, training,
# evaluation) runs as a *subprocess* using the upstream repositories, so the
# kernel itself never imports torch or numpy. That matters because CELL 5
# installs numpy<2 and pins omegaconf/hydra for the 2021-era fairseq code: the
# subprocesses pick those pins up cleanly while the kernel stays untouched.

import csv
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
from types import SimpleNamespace

print("All libraries imported successfully")


In [ ]:
# CELL 2 - Configuration: full LAV-DF test split
#
# This notebook scores AVH-Align on the COMPLETE LAV-DF test split (26,100
# clips, 6,906 real / 19,194 fake) instead of a subsample, so AP sits on the
# dataset's own class prior and is directly comparable with published numbers.
#
# It cannot run in one Kaggle session: 26,100 clips need ~28 h of mouth-ROI
# preprocessing and 37 GiB of features, against a 12 h / 20 GB session. The work
# is therefore split by MODE, and each mode resumes from the previous session's
# output:
#
#   MODE = "preprocess"  CPU only. Cuts mouth ROIs for as many not-yet-done
#                        clips as the session budget and disk allow (~10,000),
#                        and leaves them in the output. No GPU quota is spent.
#                        Run this 3x, attaching the previous outputs each time.
#
#   MODE = "score"       GPU. Attaches every ROI output, then walks the split in
#                        chunks: extract features -> score both checkpoints ->
#                        write per-clip scores -> delete the features. Peak disk
#                        stays ~17 GiB and the whole split costs ~2.5 h of GPU.
#                        The last session prints AP / AUC with bootstrap CIs.

MODE = "preprocess"        # "preprocess" (CPU sessions) then "score" (GPU session)

WORK = Path("/kaggle/working")
REPO = WORK / "AVH-Align"
AVHUBERT = WORK / "av_hubert" / "avhubert"

LINKS = WORK / "lavdf_root"
META = WORK / "lavdf_meta"
PRE = WORK / "lavdf_pre"
FEATS = WORK / "lavdf_feats"
CKPT = WORK / "checkpoints"
SCORES = WORK / "scores"          # per-chunk score CSVs; tiny, always persisted
STATE = WORK / "state"

ALL_STAGES = ["setup", "metadata", "preprocess", "extract", "train", "eval"]

CFG = SimpleNamespace(
    lavdf_root="/kaggle/input/localized-audio-visual-deepfake-dataset-lav-df",
    stages="all",
    force="",
    name="AVH-Align_LAVDF",          # our retrained checkpoint
    official="AVH-Align_AV1M",       # the authors' released checkpoint
    max_train=0, max_val=0, max_test=0,   # unused here; the whole split is scored
    epochs=0,
    budget_hours=11.0,
    workers=4,
    seed=42,
    skip_pip=False,
    chunk=4000,                      # clips per extract+score chunk
    prep_batch=2000,                 # clips per preprocess subprocess call
)

T0 = time.time()
DEADLINE = T0 + CFG.budget_hours * 3600
BYTES_PER_CLIP = None

# Measured on this pipeline: 3.9 s/clip preprocessing on 4 workers, 0.25 s/clip
# extraction on a T4, 0.28 MiB of mouth ROI and 1.46 MiB of features per clip.
# The whole test split's ROIs are only ~7 GiB, so preprocessing is bounded by
# session time, not disk; the features are what force chunked scoring.
SEC_PREP, SEC_EXTRACT = 3.9, 0.25
BYTES_ROI, BYTES_FEAT = 400_000, 1_600_000

print(f"MODE={MODE}  chunk={CFG.chunk}  budget={CFG.budget_hours}h")
print("Deadline:", time.strftime("%H:%M:%S", time.localtime(DEADLINE)))

In [ ]:
# CELL 3 - Helper Functions
#
# Small utilities shared by every stage:
#   log / run          timestamped logging and subprocess execution that aborts
#                      the run on a non-zero exit code
#   done / mark        stage completion markers in /kaggle/working/state
#   find_input_dir / fetch_repo
#                      git clone with retries, falling back to a checkout found
#                      in an attached input when GitHub is unreachable
#   budget_*, require_time, require_disk
#                      guards so a stage never starts work it cannot finish
#                      inside the session budget or the 20 GB output quota
#   run_stage          the stage runner: skips a stage whose marker exists
#                      (unless it is listed in CFG.force), otherwise runs it
#                      with timing

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def run(cmd, cwd=None, check=True):
    log(f"$ {' '.join(str(c) for c in cmd)}" + (f"   (cwd={cwd})" if cwd else ""))
    r = subprocess.run([str(c) for c in cmd], cwd=cwd)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed with code {r.returncode}")
    return r.returncode


def done(stage):
    return (STATE / f"{stage}.done").exists()


def mark(stage):
    STATE.mkdir(parents=True, exist_ok=True)
    (STATE / f"{stage}.done").write_text(time.ctime())


def disk_report():
    total, used, free = shutil.disk_usage(WORK)
    log(f"disk /kaggle/working: {used/2**30:.1f} GiB used, {free/2**30:.1f} GiB free")


def find_input_dir(name, must_have=(), root=Path("/kaggle/input")):
    """Locate a directory called `name` inside any attached input (dataset or
    notebook output), 1-4 levels deep, that also contains every relative path in
    `must_have`. Lets a run reuse a repo checkout persisted by an earlier run
    when GitHub is unreachable from the Kaggle worker."""
    for pat in (f"*/{name}", f"*/*/{name}", f"*/*/*/{name}", f"*/*/*/*/{name}"):
        for c in sorted(root.glob(pat)):
            if c.is_dir() and all((c / m).exists() for m in must_have):
                return c
    return None


def fetch_repo(dst, url, must_have=(), submodules=False, attempts=3):
    """git clone `url` into `dst`, retrying on transient network failures. If
    GitHub stays unreachable, fall back to a copy of the same checkout found in
    an attached input (the V8-output dataset carries AVH-Align and av_hubert).
    Every later source patch is idempotent, so a pre-patched copy is fine."""
    dst = Path(dst)
    if dst.exists():
        return
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    for i in range(attempts):
        cmd = ["git", "clone"] + ([] if submodules else ["--depth", "1"]) + [url, str(dst)]
        log(f"$ {' '.join(cmd)}   (attempt {i + 1}/{attempts})")
        r = subprocess.run(cmd, env=env)
        if r.returncode == 0:
            if submodules:
                run(["git", "submodule", "init"], cwd=dst)
                run(["git", "submodule", "update"], cwd=dst)
            return
        shutil.rmtree(dst, ignore_errors=True)
        if i + 1 < attempts:
            time.sleep(20)
    src = find_input_dir(dst.name, must_have)
    if src is None:
        raise SystemExit(f"git clone of {url} failed {attempts}x and no copy of "
                         f"{dst.name}/ with {list(must_have)} is attached as an input")
    log(f"GitHub unreachable -> copying {dst.name} from attached input {src}")
    shutil.copytree(src, dst, symlinks=True)
    log(f"copied {dst.name}: {sum(1 for _ in dst.rglob('*'))} entries")


def est_feature_bytes(n_clips, frames=190):
    # 190 frames/clip is calibrated from a measured run (6.12 GiB of .npz for
    # 4300 clips = 1.46 MiB/clip); the old default of 100 underestimated the
    # feature footprint by 2x and made the disk guard useless.
    return n_clips * frames * 2 * 1024 * 4


def time_left():
    return DEADLINE - time.time() if DEADLINE else float("inf")


def hms(sec):
    sec = max(0, int(sec))
    return f"{sec//3600}h{(sec%3600)//60:02d}m"


def budget_report(label=""):
    log(f"[budget] elapsed {hms(time.time()-T0)} | remaining {hms(time_left())} {label}")


def require_time(need_sec, what):

    if time_left() < need_sec:
        raise SystemExit(
            f"STOPPING BEFORE {what}: needs ~{hms(need_sec)} but only {hms(time_left())} "
            f"left in the budget.\nCompleted stages are marked -- rerun the same command "
            f"in a fresh session to resume.")


def free_bytes(path=WORK):
    return shutil.disk_usage(path).free


def require_disk(need_bytes, what, path=WORK):
    free = free_bytes(path)
    log(f"[disk] {what} needs ~{need_bytes/2**30:.1f} GiB, {free/2**30:.1f} GiB free at {path}")
    if free < need_bytes * 1.25:
        raise SystemExit(
            f"NOT ENOUGH DISK for {what}: needs ~{need_bytes/2**30:.1f} GiB "
            f"(+25% margin), only {free/2**30:.1f} GiB free.\n"
            f"Lower --max_train / --max_test and rerun.")


def count_rows(csv_path):
    with open(csv_path) as f:
        return sum(1 for _ in csv.DictReader(f))


ACTIVE_STAGES = ALL_STAGES if CFG.stages == "all" else [s.strip() for s in CFG.stages.split(",")]
FORCED = {s.strip() for s in CFG.force.split(",") if s.strip()}


def run_stage(name, stage_fn):
    if name not in ALL_STAGES:
        raise SystemExit(f"unknown stage {name!r}; valid: {ALL_STAGES}")
    if name not in ACTIVE_STAGES:
        log(f"--- {name} not selected in CFG.stages; skipping ---")
        return
    if done(name) and name not in FORCED:
        log(f"--- skipping {name} (already done; add it to CFG.force to re-run) ---")
        return
    log(f"=== stage: {name} (remaining budget {hms(time_left())}) ===")
    t0 = time.time()
    stage_fn(CFG)
    log(f"=== {name} finished in {(time.time()-t0)/60:.1f} min ===")


log(f"[budget] wall-clock budget {CFG.budget_hours:.1f}h "
    f"(Kaggle hard limit is 12h); deadline at "
    f"{time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
est = est_feature_bytes(CFG.max_train + CFG.max_val + CFG.max_test)
log(f"[plan] max_train={CFG.max_train} max_val={CFG.max_val} max_test={CFG.max_test} "
    f"-> ~{est/2**30:.2f} GiB of features, plus ~4 GiB env+checkpoint")


In [ ]:
# CELL 4 - Resume: checkpoints, mouth ROIs and scores from earlier sessions
#
# Three kinds of prior work can be attached as inputs, and each is picked up
# independently:
#   checkpoints/<name>.pt   the trained alignment head (from the training run)
#   lavdf_pre/val/*_roi.mp4 mouth ROIs cut by earlier "preprocess" sessions
#   scores/chunk_*.csv      per-clip scores already computed
# Nothing is recomputed that is already present, so a session always makes
# forward progress even if an earlier one was cut short.

def restore_inputs():
    inp = Path("/kaggle/input")
    if not inp.exists():
        log("[resume] no inputs attached")
        return

    ck = find_input_dir("checkpoints", (f"{CFG.name}.pt",))
    if ck is not None and not (CKPT / f"{CFG.name}.pt").exists():
        CKPT.mkdir(parents=True, exist_ok=True)
        shutil.copy(ck / f"{CFG.name}.pt", CKPT / f"{CFG.name}.pt")
        log(f"[resume] trained checkpoint <- {ck}")

    # Scores are small: copy them in so this session can skip those chunks and
    # still write one complete score set to its own output.
    SCORES.mkdir(parents=True, exist_ok=True)
    n_sc = 0
    for cur, dirs, files in os.walk(inp, topdown=True):
        if Path(cur).name == "scores":
            for f in files:
                if f.startswith("chunk_") and f.endswith(".csv"):
                    dst = SCORES / f
                    if not dst.exists():
                        shutil.copy(Path(cur) / f, dst)
                        n_sc += 1
            dirs[:] = []
        elif len(dirs) + len(files) > 2000 or cur[len(str(inp)):].count(os.sep) >= 6:
            dirs[:] = []
    if n_sc:
        log(f"[resume] {n_sc} score chunks copied from inputs")

    # ROI directories stay where they are (read-only mounts); a symlink farm in
    # CELL 6 gives the extraction script one directory that spans all of them.
    global ROI_MOUNTS
    ROI_MOUNTS = []
    for cur, dirs, files in os.walk(inp, topdown=True):
        p = Path(cur)
        if p.name == "val" and p.parent.name == "lavdf_pre":
            ROI_MOUNTS.append(p)
            dirs[:] = []
        elif len(dirs) + len(files) > 2000 or cur[len(str(inp)):].count(os.sep) >= 7:
            dirs[:] = []
    if ROI_MOUNTS:
        log(f"[resume] mouth-ROI mounts: {[str(m) for m in ROI_MOUNTS]}")
    else:
        log("[resume] no mouth ROIs attached")


ROI_MOUNTS = []
restore_inputs()

In [ ]:
# CELL 5 - Environment Setup: Repositories, Dependencies, Compatibility Patches, Assets
#
# What this cell does, in order:
#   0. Checks the accelerator first. The pinned torch build drives sm_70 and
#      newer; on Kaggle's P100 (sm_60) every CUDA kernel raises and the upstream
#      extraction script swallows it, so 4300 clips yield zero features after
#      hours of work. That is a two-minute failure here instead.
#   1. Clones bit-ml/AVH-Align and facebookresearch/av_hubert (with its pinned
#      fairseq submodule, commit afc77bd from 2021). The clone is retried, and
#      if GitHub is unreachable from the worker the checkout persisted in the
#      attached V8-output dataset is copied instead (fetch_repo, CELL 3).
#   2. Installs the pinned dependencies (numpy<2, omegaconf 2.0.6, hydra 1.0.7,
#      dlib-bin, python_speech_features, ...). fairseq is put on sys.path via a
#      .pth file instead of pip, because its CUDA extension no longer builds
#      against current torch and is not needed here.
#   3. Forces torch.load(weights_only=False) for every subprocess so the 2021
#      AV-HuBERT checkpoint (pickled config objects) can be loaded on torch 2.x.
#   4. Patches the 2021 code for Python 3.12 / numpy >= 1.24: mutable dataclass
#      defaults -> field(default_factory=...), removed numpy aliases, moved
#      collections ABCs, and rewrites av_hubert's package-relative imports to
#      absolute ones (the extraction script runs them as top-level modules).
#      fairseq.hydra_init() is disabled because the hydra CLI is never used.
#   5. Runs a self-healing import check that loads the *exact* module chain the
#      feature-extraction script needs, so any remaining incompatibility fails
#      here in minutes instead of after hours of preprocessing.
#   6. Downloads the dlib 68-landmark model, the mean-face template and the
#      AV-HuBERT large checkpoint (self_large_vox_433h.pt, 5.4 GB), copies the
#      deepfake_* scripts into place, and removes their unused multimodal pass
#      (~33% less GPU time and feature disk).

# ffmpeg is hard-coded by the preprocessing script; make sure it is installed.
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)


def stage_setup(args):

    WORK.mkdir(parents=True, exist_ok=True)

    # The torch build in this image supports sm_70 and newer (T4, V100, A100,
    # L4). Kaggle's P100 is sm_60: .cuda() succeeds but every kernel launch
    # raises, and the upstream extraction script swallows that in a bare
    # `except:` -- so all 4300 clips report "Unprocessed" and produce nothing
    # after hours of work (exactly how the 2026-09-03 run died). Check it here,
    # where the run has cost two minutes instead of five hours.
    probe = subprocess.run(
        [sys.executable, "-c",
         "import torch\n"
         "if not torch.cuda.is_available():\n"
         "    print('GPU none')\n"
         "else:\n"
         "    print('GPU', torch.cuda.get_device_name(0), *torch.cuda.get_device_capability(0))"],
        capture_output=True, text=True)
    line = (probe.stdout or "").strip().splitlines()[-1] if probe.stdout.strip() else "GPU unknown"
    log(f"[gpu] {line}")
    parts = line.split()
    if len(parts) >= 3 and parts[-2].isdigit():
        major = int(parts[-2])
        if major < 7:
            raise SystemExit(
                f"UNUSABLE ACCELERATOR: {' '.join(parts[1:-2])} is compute capability "
                f"{parts[-2]}.{parts[-1]}; this torch build supports 7.0 and newer.\n"
                "Feature extraction would silently produce zero features.\n"
                "Fix: Settings -> Accelerator -> GPU T4 x2, then re-run.")
    elif line == "GPU none":
        raise SystemExit("no CUDA device visible -- enable a GPU accelerator and re-run")

    # Clone with retries; if GitHub is unreachable from the worker (V4 died on
    # "could not read Username for 'https://github.com'"), reuse the checkout
    # persisted in the attached V8-output dataset instead.
    fetch_repo(REPO, "https://github.com/bit-ml/AVH-Align.git",
               must_have=("train.py", "eval.py", "deepfake_preprocess.py"))
    fetch_repo(WORK / "av_hubert", "https://github.com/facebookresearch/av_hubert.git",
               must_have=("avhubert/hubert.py", "fairseq/fairseq/checkpoint_utils.py"),
               submodules=True)

    HEAL = None
    if not args.skip_pip:


        pins = [
            "numpy<2", "omegaconf==2.0.6", "hydra-core==1.0.7",
            "sacrebleu<2.0", "bitarray", "editdistance",
            "python_speech_features", "scikit-video", "librosa",
            "dlib-bin", "cython<3", "wheel",
            "setuptools<70",
        ]
        run([sys.executable, "-m", "pip", "install", "-q", "pip<24.1"])
        run([sys.executable, "-m", "pip", "install", "-q", *pins])
        fsq = WORK / "av_hubert" / "fairseq"
        import site, re as _re
        sp = next(pp for pp in site.getsitepackages() if "packages" in pp)
        (Path(sp) / "fairseq_local.pth").write_text(str(fsq) + "\n")
        log("fairseq installed by .pth path (pip C build skipped: libnat_cuda "
            "needs THC/THC.h, removed from torch -- unbuildable and unneeded)")

        # torch >= 2.6 defaults weights_only=True, which breaks loading the 2021
        # AV-HuBERT checkpoint (pickled fairseq Dictionary / argparse objects).
        # V8 lesson: a sitecustomize.py in site-packages is NEVER imported here --
        # Debian ships /usr/lib/python3.x/sitecustomize.py earlier on sys.path and
        # Python loads only the first one. Three independent layers instead:
        #   1. env var TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 in this process, inherited
        #      by every subprocess (torch honours it whenever weights_only is not
        #      passed explicitly);
        #   2. a .pth "import" line, executed at site init by every interpreter that
        #      uses this site-packages, setting the same env var;
        #   3. every single-line torch.load(...) in fairseq + avhubert rewritten to
        #      pass weights_only=False explicitly, so the fix does not depend on the
        #      environment at all.
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
        (Path(sp) / "sitecustomize.py").unlink(missing_ok=True)
        (Path(sp) / "torch_weights_only_off.pth").write_text(
            "import os; os.environ.setdefault('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD', '1')\n")
        nwo = 0
        for f in list(fsq.rglob("*.py")) + list(AVHUBERT.glob("*.py")):
            try:
                t = f.read_text()
            except Exception:
                continue
            t2 = _re.sub(r"torch\.load\(((?:[^()]|\([^()]*\))*)\)",
                         lambda m: m.group(0) if "weights_only" in m.group(1)
                         else f"torch.load({m.group(1)}, weights_only=False)", t)
            if t2 != t:
                f.write_text(t2)
                nwo += 1
        log(f"torch.load weights_only=False: env var + .pth hook set, "
            f"{nwo} fairseq/avhubert files patched explicitly")

        # ---- self-healing dataclass patch ---------------------------------
        # fairseq afc77bd + hydra 1.0.7 + omegaconf 2.0.6 predate py3.11's strict
        # dataclass rule (mutable default `x: T = T()` must use default_factory).
        # Rather than guess every offending module, loop: try `import fairseq`,
        # and whenever it dies on a mutable-default error, read the exact file from
        # the traceback, wrap that pattern, and retry. This walks fairseq -> hydra
        # -> omegaconf -> whatever is next, automatically.
        DC_PAT = _re.compile(r"(\w+): ([A-Za-z_][\w.]*) = \2\(\)")
        FROM_DC = _re.compile(r"from dataclasses import (?![^\n]*\bfield\b)")
        NP_ALIAS = {"float": "float", "int": "int", "bool": "bool",
                    "object": "object", "str": "str", "complex": "complex",
                    "long": "int", "unicode": "str"}
        ABC_NAMES = ("Mapping", "MutableMapping", "Sequence", "MutableSequence",
                     "Set", "MutableSet", "Iterable", "Callable", "Hashable",
                     "Container", "Sized")

        def _patch_file(fp):
            # Apply every known 2021-code -> modern-runtime fix and report if the
            # file changed. Covers: py3.12 mutable-default dataclasses, numpy>=1.24
            # removed aliases (np.float/int/bool/object/str/...), and py3.10 moved
            # collections ABCs. Idempotent: re-running makes no further change.
            try:
                t = fp.read_text()
            except Exception:
                return False
            o = t
            t = DC_PAT.sub(r"\1: \2 = field(default_factory=\2)", t)
            if t != o and "field(default_factory=" in t:
                if "from dataclasses import" in t and FROM_DC.search(t):
                    t = FROM_DC.sub("from dataclasses import field, ", t, count=1)
                elif "from dataclasses import" not in t:
                    t = "from dataclasses import field\n" + t
            for pfx in ("np", "numpy"):
                for k, v in NP_ALIAS.items():
                    t = _re.sub(r"\b" + pfx + r"\." + k + r"\b", v, t)
            for nm in ABC_NAMES:
                t = _re.sub(r"\bcollections\." + nm + r"\b",
                            "collections.abc." + nm, t)
            if t == o:
                return False
            fp.write_text(t)
            return True

        n0 = 0
        for root in (fsq / "fairseq", WORK / "av_hubert" / "avhubert",
                     Path(sp) / "hydra", Path(sp) / "omegaconf"):
            for f in root.rglob("*.py"):
                if _patch_file(f):
                    n0 += 1
        log(f"default_factory pre-patch: {n0} files")

        # av_hubert's avhubert/*.py use package-relative imports (e.g.
        # "from .hubert_dataset import AVHubertDataset"). The deepfake_* entry
        # scripts run as top-level scripts (cwd=avhubert) and do absolute
        # "import hubert_pretraining", so those relative imports raise
        # "attempted relative import with no known parent package". avhubert is
        # on sys.path via cwd, so rewrite relative -> absolute.
        nrel = 0
        for f in AVHUBERT.glob("*.py"):
            t = o2 = f.read_text()
            t = _re.sub(r'(?m)^(\s*)from \. import ', r'\1import ', t)
            t = _re.sub(r'(?m)^(\s*)from \.([A-Za-z_])', r'\1from \2', t)
            if t != o2:
                f.write_text(t)
                nrel += 1
        log(f"rel-import fix: {nrel} avhubert modules -> absolute imports")

        # THE CORE CONTRADICTION (verified): py3.12 forbids `x: T = T()` mutable
        # defaults -> we rewrite them to field(default_factory=T). But omegaconf
        # 2.0.6's OmegaConf.structured() reads field.default, which for a
        # default_factory field is dataclasses.MISSING -> "_MISSING_TYPE" error.
        # These cannot both be satisfied by patching. structured() is only called
        # from fairseq.hydra_init(), which registers configs for the hydra CLI we
        # never use -- feature extraction calls fairseq APIs directly and reads the
        # checkpoint's cfg as DATA. So we neuter hydra_init: the schemas still
        # define (default_factory), structured() is never invoked on them.
        finit = fsq / "fairseq" / "__init__.py"
        it = finit.read_text()
        if "hydra_init()" in it and "FAIRSEQ_HYDRA_INIT" not in it:
            it = it.replace(
                "hydra_init()",
                "import os as _os\n"
                "if _os.environ.get('FAIRSEQ_HYDRA_INIT') == '1':\n"
                "    hydra_init()", 1)
            finit.write_text(it)
            log("neutered fairseq.hydra_init() (avoids omegaconf structured()/MISSING)")

        # Exclude the *raising* machinery so the picker targets the caller
        # (the file that actually uses the bad construct), not the module
        # that raised (numpy/__init__.py __getattr__, dataclasses.py, ...).
        _STDLIB = ("/dataclasses.py", "/enum.py", "/typing.py",
                   "/functools.py", "/numpy/__init__.py", "/abc.py",
                   "/collections/__init__.py", "/re/__init__.py")
        # Deeper than `import fairseq`: exercise the actual extraction path so a
        # green light really means the pipeline can load models, not just import.
        # Also import the avhubert model modules the extraction script needs,
        # with cwd=avhubert exactly like the real run -- so a broken import
        # chain fails here in setup (minutes), not after hours of preprocess.
        # The torch.save/torch.load round trip of a fairseq Dictionary is exactly
        # what checkpoint_utils.load_checkpoint_to_cpu does on the 5.4 GB AV-HuBERT
        # file; V8 only discovered it was broken 5 hours in. Now it is a 6-minute
        # setup failure instead.
        CHECK = ("import fairseq; "
                 "from fairseq import checkpoint_utils, tasks, utils; "
                 "import omegaconf, hydra; "
                 "import hubert_pretraining, hubert, hubert_asr; "
                 "import torch, tempfile, os; "
                 "from fairseq.data.dictionary import Dictionary; "
                 "p = tempfile.mktemp(suffix='.pt'); "
                 "torch.save({'d': Dictionary(), 'cfg': {'x': 1}}, p); "
                 "torch.load(p, map_location='cpu'); os.remove(p); "
                 "print('imports ok, torch.load default-args ok:', omegaconf.__version__)")
        def heal(check_code, label):
            for attempt in range(30):
                r = subprocess.run([sys.executable, "-c", check_code],
                                   capture_output=True, text=True, cwd=str(AVHUBERT))
                if r.returncode == 0:
                    out = r.stdout.strip().splitlines()
                    log(out[-1] if out else f"{label} ok")
                    return
                err = r.stderr
                paths = _re.findall(r'File "([^"]+\.py)"', err)
                target = None
                for pp in reversed(paths):
                    if any(sfx in pp for sfx in _STDLIB):
                        continue
                    target = pp
                    break
                # With cwd=avhubert a traceback may name a module by a relative
                # path; resolve it against AVHUBERT before patching.
                tp = Path(target) if target else None
                if tp is not None and not tp.is_absolute():
                    tp = AVHUBERT / tp
                if tp is None or not tp.exists() or not _patch_file(tp):
                    log(f"cannot auto-fix {label} (attempt {attempt}); last error:")
                    print(err[-3000:], flush=True)
                    break
                log(f"patched {tp} (attempt {attempt}); retrying {label}")
            raise SystemExit(f"{label} could not be healed -- see errors above")

        heal(CHECK, "fairseq/avhubert import check")
        HEAL = heal


    misc = AVHUBERT / "content" / "data" / "misc"
    misc.mkdir(parents=True, exist_ok=True)

    def stage_asset(src_glob, dest, url, rel, is_bz2=False):


        if dest.exists() and dest.stat().st_size > 1024:
            log(f"already present: {dest.name} ({dest.stat().st_size/2**20:.1f} MiB)")
            return
        # Look for a copy in an attached input before downloading. Both lookups
        # are bounded-depth: `*/**/name` would recursively walk the 136k-file
        # LAV-DF mount and can take minutes per asset for nothing.
        base = find_input_dir("av_hubert", (rel,))
        if base is not None:
            hits = [base / rel]
        else:
            name = src_glob.rsplit("/", 1)[-1]
            hits = [h for pat in (f"*/{name}", f"*/*/{name}", f"*/*/*/{name}",
                                  f"*/*/*/*/{name}", f"*/*/*/*/*/{name}")
                    for h in sorted(Path("/kaggle/input").glob(pat)) if h.is_file()]
        if hits:
            log(f"staging {hits[0]} -> {dest}")
            shutil.copy(hits[0], dest)
            return
        log(f"downloading {url}")
        if is_bz2:
            tmp = Path("/tmp/_asset.bz2")
            run(["wget", "-q", url, "-O", str(tmp)])
            import bz2 as _bz2
            dest.write_bytes(_bz2.decompress(tmp.read_bytes()))
            tmp.unlink()
        else:
            run(["wget", "-q", url, "-O", str(dest)])
        if not dest.exists() or dest.stat().st_size < 1024:
            raise SystemExit(f"download failed or truncated: {dest}")
        log(f"got {dest.name} ({dest.stat().st_size/2**20:.1f} MiB)")

    stage_asset("*/**/shape_predictor_68_face_landmarks.dat",
                misc / "shape_predictor_68_face_landmarks.dat",
                "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2",
                rel="avhubert/content/data/misc/shape_predictor_68_face_landmarks.dat",
                is_bz2=True)
    stage_asset("*/**/20words_mean_face.npy", misc / "20words_mean_face.npy",
                "https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks"
                "/raw/master/preprocessing/20words_mean_face.npy",
                rel="avhubert/content/data/misc/20words_mean_face.npy")
    stage_asset("*/**/self_large_vox_433h.pt", AVHUBERT / "self_large_vox_433h.pt",
                "https://dl.fbaipublicfiles.com/avhubert/model/lrs3_vox/vsr/"
                "self_large_vox_433h.pt",
                rel="avhubert/self_large_vox_433h.pt")

    # Load the real 5.4 GB checkpoint through the exact fairseq call the
    # extraction script makes (checkpoint_utils.load_model_ensemble_and_task).
    # Extraction has never run end-to-end in this environment (V8 died before
    # it), so its riskiest step must fail here, in setup, not hours later.
    if HEAL is not None:
        HEAL("from fairseq import checkpoint_utils; "
             "import hubert_pretraining, hubert, hubert_asr; "
             "m, _, t = checkpoint_utils.load_model_ensemble_and_task(['self_large_vox_433h.pt']); "
             "print('checkpoint load ok:', type(m[0]).__name__, 'crop', t.cfg.image_crop_size)",
             "AV-HuBERT checkpoint load check")


    for f in ("deepfake_preprocess.py", "deepfake_feature_extraction.py"):
        shutil.copy(REPO / f, AVHUBERT / f)


    fe = AVHUBERT / "deepfake_feature_extraction.py"
    src_fe = fe.read_text()
    subs = [
        ('f_mm, _ = model.extract_finetune({"video": frames, "audio": audio}, None, None)',
         'f_mm = None  # [patched] never read downstream'),
        ('f_mm.squeeze(0).cpu().numpy()', 'None'),
        ('"multimodal": feature_multimodal,', '# [patched] multimodal dropped'),
    ]
    applied = 0
    for old, new in subs:
        if old in src_fe:
            src_fe = src_fe.replace(old, new)
            applied += 1
    fe.write_text(src_fe)
    if applied == len(subs):
        log("patched feature extraction: multimodal pass removed "
            "(~33% less GPU time and feature disk)")
    else:
        log(f"WARNING: multimodal patch applied {applied}/{len(subs)} substitutions -- "
            f"upstream may have changed. Continuing with the unpatched behaviour; "
            f"expect ~50% more feature disk than estimated.")


    if not Path("/usr/bin/ffmpeg").exists():
        found = shutil.which("ffmpeg")
        if not found:
            raise SystemExit("ffmpeg not found. Run: !apt-get -qq install -y ffmpeg")
        log(f"symlinking {found} -> /usr/bin/ffmpeg (hardcoded in the repo)")
        os.symlink(found, "/usr/bin/ffmpeg")

    disk_report()
    mark("setup")


run_stage("setup", stage_setup)


In [ ]:
# CELL 6 - The complete LAV-DF test split
#
# Every clip of the official test split, in a fixed order (sorted by file path),
# so chunk boundaries are identical in every session and on every machine. No
# sampling and no class balancing: the label mix is the dataset's own, which is
# what makes AP comparable with published numbers.

def find_lavdf_metadata(root: Path) -> Path:
    """Locate LAV-DF's metadata.json, honouring an explicit root first and then
    searching the attached inputs (the mount slug varies by dataset owner)."""
    for name in ("metadata.json", "metadata.min.json"):
        for c in [root / name, *root.glob(f"*/{name}"), *root.glob(f"*/*/{name}")]:
            if c.is_file():
                return c
    inp = Path("/kaggle/input")
    pats = [f"{d}/{n}" for n in ("metadata.json", "metadata.min.json")
            for d in ("*", "*/*", "*/*/*", "*/*/*/*")]
    for pat in pats:
        for c in sorted(inp.glob(pat)):
            if "avhalign" not in str(c).lower() and c.is_file():
                return c
    raise SystemExit("no LAV-DF metadata.json found under /kaggle/input")


def build_full_test(args):
    lavdf = Path(args.lavdf_root)
    META.mkdir(parents=True, exist_ok=True)
    meta_path = find_lavdf_metadata(lavdf)
    log(f"LAV-DF metadata: {meta_path}")
    video_root = meta_path.parent

    with open(meta_path) as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = data.get("clips", list(data.values()))

    recs = []
    for r in data:
        if r.get("split") != "test":
            continue
        rel = r.get("file") or r.get("path") or r.get("filename")
        nf = r.get("video_frames") or r.get("n_frames") or r.get("num_frames")
        if rel is None or not nf or int(nf) < 31:
            continue
        mv, ma = bool(r.get("modify_video")), bool(r.get("modify_audio"))
        nfakes = r.get("n_fakes", len(r.get("fake_periods") or []))
        recs.append({"rel": str(rel), "num_frames": int(nf),
                     "label": 0 if (not mv and not ma and nfakes == 0) else 1})

    recs.sort(key=lambda r: r["rel"])          # fixed order == stable chunks
    seen = {}
    for rc in recs:
        base = Path(rc["rel"]).name
        if base in seen and seen[base] != rc["rel"]:
            base = Path(rc["rel"]).parent.name + "_" + base
        seen[base] = rc["rel"]
        rc["flat"] = base

    n_real = sum(1 for r in recs if r["label"] == 0)
    log(f"[test] full LAV-DF test split: {len(recs)} clips, {n_real} real / "
        f"{len(recs) - n_real} fake (fake prior {1 - n_real / len(recs):.3f})")

    with open(META / "full_test.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["path", "label", "num_frames", "rel"])
        w.writeheader()
        for rc in recs:
            w.writerow({"path": rc["flat"], "label": rc["label"],
                        "num_frames": rc["num_frames"], "rel": rc["rel"]})
    log(f"wrote {META / 'full_test.csv'}  rows={len(recs)}")
    return recs, video_root, lavdf


def link_clips(recs, video_root, lavdf):
    """Symlink farm in the layout deepfake_preprocess.py expects (test -> val)."""
    (LINKS / "val").mkdir(parents=True, exist_ok=True)
    n = 0
    for rc in recs:
        src = video_root / rc["rel"]
        if not src.exists():
            src = lavdf / rc["rel"]
        dst = LINKS / "val" / rc["flat"]
        if dst.exists() or dst.is_symlink() or not src.exists():
            continue
        os.symlink(src, dst)
        n += 1
    log(f"symlinked {n} clips -> {LINKS / 'val'}")


def roi_done(recs):
    """Names whose mouth ROI already exists in an attached mount or locally."""
    have = set()
    for m in ROI_MOUNTS + [PRE / "val"]:
        if m.exists():
            have |= {p.name[:-8] for p in m.glob("*_roi.mp4")}   # strip _roi.mp4
    return {rc["flat"] for rc in recs if rc["flat"][:-4] in have}


TEST_RECS, VIDEO_ROOT, LAVDF_ROOT = build_full_test(CFG)

In [ ]:
# CELL 7 - Mode "preprocess": cut mouth ROIs, CPU only
#
# Takes the next clips that have no ROI yet and preprocesses as many as the
# session's time and disk allow, in batches of CFG.prep_batch so an interrupted
# session still leaves every completed batch in the output. Nothing here needs a
# GPU, so run these sessions with the accelerator set to None and keep the GPU
# quota for the scoring session.

def stage_prep(args):
    done = roi_done(TEST_RECS)
    todo = [rc for rc in TEST_RECS if rc["flat"] not in done]
    log(f"[prep] {len(done)}/{len(TEST_RECS)} clips already have ROIs; {len(todo)} left")
    if not todo:
        log("[prep] nothing to do -- switch MODE to 'score'")
        return

    by_time = int(max(0, time_left() - 20 * 60) / SEC_PREP)
    by_disk = int(free_bytes(WORK) * 0.85 / BYTES_ROI)
    n = max(0, min(len(todo), by_time, by_disk))
    log(f"[prep] budget allows {by_time} clips, disk allows {by_disk} -> doing {n}")
    if n == 0:
        raise SystemExit("no room for even one clip; attach fewer inputs or a longer session")

    batch = todo[:n]
    link_clips(batch, VIDEO_ROOT, LAVDF_ROOT)
    PRE.mkdir(parents=True, exist_ok=True)

    for i in range(0, len(batch), args.prep_batch):
        part = batch[i:i + args.prep_batch]
        if time_left() < len(part) * SEC_PREP + 10 * 60:
            log(f"[prep] stopping before batch {i // args.prep_batch}: not enough budget left")
            break
        csv_path = META / f"prep_{i // args.prep_batch}.csv"
        with open(csv_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["path", "label"])
            w.writeheader()
            for rc in part:
                w.writerow({"path": rc["flat"], "label": rc["label"]})
        run([sys.executable, "deepfake_preprocess.py",
             "--dataset", "AV1M", "--split", "test",
             "--metadata", csv_path,
             "--data_path", LINKS,
             "--save_path", PRE,
             "--max_workers", str(args.workers)], cwd=AVHUBERT)
        have = len(list((PRE / "val").glob("*_roi.mp4"))) if (PRE / "val").exists() else 0
        log(f"[prep] {have} ROIs in this session's output")
        disk_report()
        budget_report(f"after batch {i // args.prep_batch}")

    total = len(roi_done(TEST_RECS))
    log(f"[prep] {total}/{len(TEST_RECS)} clips now have ROIs "
        f"({len(TEST_RECS) - total} still to do)")
    if total >= len(TEST_RECS):
        log("[prep] the split is fully preprocessed -- next session: MODE = 'score'")

In [ ]:
# CELL 8 - Mode "score": extract features and score both checkpoints
#
# For each chunk of CFG.chunk clips: run AV-HuBERT over the chunk's mouth ROIs,
# score every clip with both checkpoints in a single pass over the features, save
# the per-clip scores, then delete the features. Peak disk stays around 17 GiB
# and only the scores survive, so the whole 26,100-clip split fits in one GPU
# session at roughly 2.5 h.
#
# The scoring code is byte-for-byte the pipeline in the authors' eval.py --
# L2-normalise both streams, run the fusion model, take logsumexp(-output) -- so
# the numbers are theirs, only the loop is ours.

SCORER = r'''
import csv, os, sys, numpy as np, torch
from model import FusionModel

meta, feats, out_csv = sys.argv[1], sys.argv[2], sys.argv[3]
ckpts = dict(p.split("=", 1) for p in sys.argv[4:])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {}
for name, path in ckpts.items():
    m = FusionModel().to(device)
    m.load_state_dict(torch.load(path, weights_only=False)["state_dict"])
    m.eval()
    models[name] = m

rows = list(csv.DictReader(open(meta)))
names = list(models)
with open(out_csv, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["path", "label"] + [f"score_{n}" for n in names])
    kept = 0
    for r in rows:
        f = os.path.join(feats, r["path"].replace(".mp4", ".npz"))
        if not os.path.exists(f):
            continue
        d = np.load(f, allow_pickle=True)
        v = torch.from_numpy(d["visual"]).to(device)
        a = torch.from_numpy(d["audio"]).to(device)
        v = v / torch.linalg.norm(v, ord=2, dim=-1, keepdim=True)
        a = a / torch.linalg.norm(a, ord=2, dim=-1, keepdim=True)
        scores = []
        with torch.no_grad():
            for n in names:
                o = models[n](v, a)
                scores.append(float(torch.logsumexp(-o, dim=0).detach().cpu().squeeze()))
        w.writerow([r["path"], r["label"]] + scores)
        kept += 1
print(f"scored {kept}/{len(rows)} clips -> {out_csv}")
'''

AGGREGATE = r'''
import csv, glob, sys, numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

rows = []
for f in sorted(glob.glob(sys.argv[1] + "/chunk_*.csv")):
    rows += list(csv.DictReader(open(f)))
if not rows:
    sys.exit("no scores found")

y = np.array([int(r["label"]) for r in rows])
names = [k[6:] for k in rows[0] if k.startswith("score_")]
rng = np.random.default_rng(0)
idx = [rng.integers(0, len(y), len(y)) for _ in range(1000)]

print(f"clips scored: {len(y)}  ({int((y == 0).sum())} real / {int(y.sum())} fake, "
      f"fake prior {y.mean():.3f})")
for n in names:
    s = np.array([float(r["score_" + n]) for r in rows])
    ap, auc = average_precision_score(y, s), roc_auc_score(y, s)
    aps = [average_precision_score(y[i], s[i]) for i in idx]
    aucs = [roc_auc_score(y[i], s[i]) for i in idx]
    lo_ap, hi_ap = np.percentile(aps, [2.5, 97.5])
    lo_au, hi_au = np.percentile(aucs, [2.5, 97.5])
    print(f"{n}:  AP {ap:.4f} [{lo_ap:.4f}, {hi_ap:.4f}]   "
          f"AUC {auc:.4f} [{lo_au:.4f}, {hi_au:.4f}]")
'''


def stage_score(args):
    if not ROI_MOUNTS and not (PRE / "val").exists():
        raise SystemExit("no mouth ROIs available -- run MODE='preprocess' sessions first")

    # One directory spanning every attached ROI mount, so the extraction script
    # sees a single --data_path. Symlinks, so nothing is copied.
    merged = PRE / "val"
    merged.mkdir(parents=True, exist_ok=True)
    n_link = 0
    for m in ROI_MOUNTS:
        for p in m.iterdir():
            dst = merged / p.name
            if not dst.exists() and not dst.is_symlink():
                os.symlink(p, dst)
                n_link += 1
    log(f"[score] {n_link} ROI symlinks merged into {merged}")

    have = {p.name[:-8] for p in merged.glob("*_roi.mp4")}
    ready = [rc for rc in TEST_RECS if rc["flat"][:-4] in have]
    log(f"[score] {len(ready)}/{len(TEST_RECS)} clips have ROIs")

    SCORES.mkdir(parents=True, exist_ok=True)
    ck = {}
    if (CKPT / f"{args.name}.pt").exists():
        ck[args.name] = CKPT / f"{args.name}.pt"
    off = REPO / "checkpoints" / f"{args.official}.pt"
    if off.exists():
        ck[args.official] = off
    if not ck:
        raise SystemExit("no checkpoint to score with")
    log(f"[score] checkpoints: {list(ck)}")
    (REPO / "score_clips.py").write_text(SCORER)

    chunks = [(i, ready[i:i + args.chunk]) for i in range(0, len(ready), args.chunk)]
    for start, part in chunks:
        tag = f"chunk_{start:06d}"
        out = SCORES / f"{tag}.csv"
        if out.exists():
            log(f"[score] {tag} already scored; skipping")
            continue
        need = len(part) * SEC_EXTRACT + 180
        if time_left() < need + 10 * 60:
            log(f"[score] stopping before {tag}: needs ~{hms(need)}, "
                f"{hms(time_left())} left. Re-run with this output attached.")
            break
        require_disk(len(part) * BYTES_FEAT, f"features for {tag}")

        meta_csv = META / f"{tag}_meta.csv"
        with open(meta_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["path", "label"])
            w.writeheader()
            for rc in part:
                w.writerow({"path": rc["flat"], "label": rc["label"]})

        run([sys.executable, "deepfake_feature_extraction.py",
             "--dataset", "AV1M", "--split", "test",
             "--metadata", meta_csv,
             "--ckpt_path", "self_large_vox_433h.pt",
             "--data_path", PRE,
             "--save_path", FEATS], cwd=AVHUBERT)

        made = len(list((FEATS / "val").glob("*.npz"))) if (FEATS / "val").exists() else 0
        log(f"[score] {tag}: {made} feature files present")
        if made == 0:
            raise SystemExit(f"{tag} produced no features -- check the accelerator")

        run([sys.executable, "score_clips.py", meta_csv, FEATS / "val", out]
            + [f"{n}={p}" for n, p in ck.items()], cwd=REPO)
        shutil.rmtree(FEATS / "val", ignore_errors=True)
        disk_report()
        budget_report(f"after {tag}")

    scored = sorted(SCORES.glob("chunk_*.csv"))
    n_done = sum(sum(1 for _ in csv.DictReader(open(f))) for f in scored)
    log(f"[score] {n_done}/{len(TEST_RECS)} clips scored across {len(scored)} chunks")

    if n_done >= len(ready) and ready:
        (REPO / "aggregate_scores.py").write_text(AGGREGATE)
        log("=== FULL TEST SPLIT RESULTS ===")
        run([sys.executable, "aggregate_scores.py", SCORES], cwd=REPO, check=False)
    else:
        log("[score] more chunks to go -- attach this output to the next session")


if MODE == "preprocess":
    run_stage("preprocess", stage_prep)
elif MODE == "score":
    run_stage("eval", stage_score)
else:
    raise SystemExit(f"unknown MODE {MODE!r}; use 'preprocess' or 'score'")

In [ ]:
# CELL 9 - Session summary
#
# Prints what this session added and what the next one should do, so a multi-
# session run never depends on remembering where it left off.

n_roi = len(roi_done(TEST_RECS))
n_scored = sum(sum(1 for _ in csv.DictReader(open(f))) for f in sorted(SCORES.glob("chunk_*.csv"))) \
    if SCORES.exists() else 0

print("=================================")
print("AVH-Align on the full LAV-DF test split")
print("=================================")
print(f"MODE this session      : {MODE}")
print(f"clips in the split     : {len(TEST_RECS)}")
print(f"mouth ROIs available   : {n_roi}")
print(f"clips scored           : {n_scored}")
disk_report()
budget_report("session end")
print()
if n_roi < len(TEST_RECS):
    print(f"NEXT: another MODE='preprocess' session (accelerator None), with this")
    print(f"      output and every earlier ROI output attached. {len(TEST_RECS) - n_roi} clips to go.")
elif n_scored < len(TEST_RECS):
    print("NEXT: MODE='score' on GPU T4 x2, with every ROI output attached")
    print("      plus this output if any chunks were already scored.")
else:
    print("DONE: AP / AUC with bootstrap CIs are printed above, on all")
    print(f"      {len(TEST_RECS)} test clips at the dataset's own class prior.")
print("=================================")
